In [2]:
# Building a sample vectordb
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [4]:
#Loading Data
loader=TextLoader("speech.txt")
docs=loader.load()

docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [17]:
#Splitting Text
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=30)
text_splits=text_splitter.split_documents(docs)
text_splits

[Document(metadata={'source': 'speech.txt'}, page_content='reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(metadata={'source': 'speech.txt'}, page_content='reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(metadata={'source': 'speech.txt'}, page_content='government in the hour of test. They are, most of them, as true and loyal Americans as if they had never known any other fealty or allegiance. They will be prompt to stand with us in rebuking and restraining the few who may be of a different mind and purpose. If there should be disloyalty, it will be dealt with with a firm hand of stern repression; but, if it lifts its head at all, it will lift it only here and there and without countenance except from a lawless and malign

In [ ]:
#Converting text into Embeddings

embedding=OllamaEmbeddings(model="gemma:2b")

#Persist_directory is used to store this vectors in sqlite3 so that we can reuse
vectordb=Chroma.from_documents(documents=text_splits,embedding=embedding,persist_directory="./chroma_db")
vectordb



In [22]:
#Querying

query="To such a task we can dedicate our lives and our fortunes"

docs=vectordb.similarity_search(query)
docs[0].page_content

'reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

In [27]:
#load from disk

db2=Chroma(persist_directory="./chroma_db",embedding_function=embedding)
docs=db2.similarity_search(query)
docs[0].page_content

'reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

In [28]:
docs=db2.similarity_search_with_score(query)
docs

[(Document(id='1f7e36f3-427f-458d-8165-6f46861aa77b', metadata={'source': 'speech.txt'}, page_content='reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  3128.30126953125),
 (Document(id='fecbed9b-de7a-49de-97f5-9ed41f2c3430', metadata={'source': 'speech.txt'}, page_content='reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  3128.30126953125),
 (Document(id='8c31e6cb-b101-425c-a908-98ba41156547', metadata={'source': 'speech.txt'}, page_content='government in the hour of test. They are, most of them, as true and loyal Americans as if they had never known any other fealty or allegiance. They will be prompt to stand with us in rebuking and restraining the few who may be of a different mind and purpose. If there should be disloyalty, it will be d

In [30]:
#Retriever Option
retriever=vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'